In [ ]:
%load_ext autoreload
%autoreload 2

In [ ]:
import nest_asyncio

nest_asyncio.apply()

In [ ]:
from textwrap import dedent

from dotenv import load_dotenv
from pydantic_ai import Agent, ModelSettings

load_dotenv()

## Non-reasoning models

Pydantic AI lets you control model parameters via `ModelSettings`. You can set `temperature`, `seed`, `max_tokens`, and more.

In [ ]:
agent = Agent(
    "openai:gpt-5-nano",
    system_prompt="You're Jose, a helpful assistant that replies in haikus.",
    model_settings=ModelSettings(temperature=0.2, seed=42),
)

for i in range(5):
    result = agent.run_sync("Hello, my name is Dylan. Who are you?")
    print(result.output)
    print()

## Thinking / Reasoning models

Some models can perform internal chain-of-thought reasoning before providing their final answer. Pydantic AI supports this via provider-specific model settings.

For OpenAI, `OpenAIResponsesModel` and `OpenAIResponsesModelSettings` let you control reasoning effort and extract reasoning summaries. See [Pydantic AI Thinking docs](https://ai.pydantic.dev/thinking/) for details.

In [ ]:
text = """
The Xbox 360 is a home video game console developed by Microsoft. As the successor to the original Xbox, it is the second console in the Xbox series. It competed with Sony's PlayStation 3 and Nintendo's Wii as part of the seventh generation of video game consoles. It was officially unveiled on MTV on May 12, 2005, with detailed launch and game information announced later that month at the 2005 Electronic Entertainment Expo (E3). This photograph shows the "Pro" model from the launch line-up, which featured a 20GB hard drive, wireless controller and a silver DVD bezel. 
"""

prompt = dedent(f"""
How many times does the word "the" (case-insensitive) appear in the following text?

Text: {text}
""")

In [ ]:
# Non-reasoning model
agent_mini = Agent(
    "openai:gpt-5-mini",
    system_prompt="You are a helpful assistant.",
)

result = agent_mini.run_sync(prompt)
print("gpt-5-mini:")
print(result.output)

In [ ]:
from pydantic_ai.models.openai import OpenAIResponsesModel, OpenAIResponsesModelSettings
from pydantic_ai.messages import ModelResponse, ThinkingPart

model = OpenAIResponsesModel("gpt-5-mini")
agent_reasoning = Agent(
    model=model,
    system_prompt="You are a helpful assistant.",
    model_settings=OpenAIResponsesModelSettings(
        openai_reasoning_effort="high",
        openai_reasoning_summary="detailed",
    ),
)

result = agent_reasoning.run_sync(prompt)
print("gpt-5-mini (reasoning):")
print(result.output)

The reasoning summary is available as `ThinkingPart` objects in the model response messages.

In [ ]:
for message in result.new_messages():
    if isinstance(message, ModelResponse) and message.thinking:
        print("Reasoning summary:")
        print(message.thinking)

# Exercise: 

1. Make the assistant reply in another language

2. Implement Chain-of-thought for non-reasoning model

3. Asked with a Yes/No question make the model only return only "Yes" or "No", and get the probability of each